In [1]:
1

1

In [2]:
import os, getpass

os.environ.setdefault("DEEPEVAL_TELEMETRY_OPT_OUT", "YES")  # skip DeepEval's anonymous telemetry

if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your GROQ_API_KEY: ")

print("Key set.")

Key set.


## Test Case

In [3]:
from deepeval.test_case import LLMTestCase

test_cases = [
    LLMTestCase(
        input="What's your refund policy?",
        actual_output=(
            "Happy to help! We offer full refunds within 30 days of purchase, no questions "
            "asked. Just reply here with your order number."
        ),
        expected_output="Refunds are available within 30 days of purchase.",
    ),
    LLMTestCase(
        input="What's your refund policy?",
        actual_output="No refunds. Read the policy page next time.",
        expected_output="Refunds are available within 30 days of purchase.",
    ),
]

## The judge: `LocalModel` (Groq)

In [4]:
from deepeval.models import LocalModel

judge = LocalModel(
    model="llama-3.3-70b-versatile",
    api_key=os.environ["GROQ_API_KEY"],
    base_url="https://api.groq.com/openai/v1",
    temperature=0,
)
print("Judge ready:", judge.get_model_name())

Judge ready: llama-3.3-70b-versatile (Local Model)


## Define two G-Eval metrics

In [5]:
from deepeval.metrics import GEval
from deepeval.test_case import SingleTurnParams

# correctness = GEval(
#     name="Correctness",
#     criteria="Determine whether the actual output is factually correct given the expected output.",
#     evaluation_params=[
#         SingleTurnParams.INPUT,
#         SingleTurnParams.ACTUAL_OUTPUT,
#         SingleTurnParams.EXPECTED_OUTPUT,
#     ],
#     model=judge,
# )

tone = GEval(
    name="Professional Tone",
    evaluation_steps=[
        "Check whether the response is polite and professional.",
        "Penalize sarcasm, rudeness, or dismissive language.",
        "Reward clear, respectful phrasing even if brief.",
    ],
    evaluation_params=[SingleTurnParams.INPUT, SingleTurnParams.ACTUAL_OUTPUT],
    model=judge,
)

In [6]:
# from deepeval.metrics.g_eval import Rubric, GEval
# from deepeval.test_case import SingleTurnParams

# correctness = GEval(
#     name="Correctness",
#     criteria="Determine whether the actual output is factually correct given the expected output.",
#     evaluation_params=[
#         SingleTurnParams.INPUT,
#         SingleTurnParams.ACTUAL_OUTPUT,
#         SingleTurnParams.EXPECTED_OUTPUT,
#     ],
#     rubric=[
#         Rubric(score_range=(0, 2), expected_outcome="Factually wrong or contradicts the expected output."),
#         Rubric(score_range=(3, 5), expected_outcome="Partially correct -- misses or garbles a key fact."),
#         Rubric(score_range=(6, 8), expected_outcome="Mostly correct with only minor omissions."),
#         Rubric(score_range=(9, 10), expected_outcome="Fully correct and complete."),
#     ],
#     model=judge,  # your existing LocalModel(Groq) judge
# )

## Run the evaluation

In [7]:
from deepeval import evaluate
from deepeval.evaluate.configs import AsyncConfig, DisplayConfig, ErrorConfig

results = evaluate(
    test_cases=test_cases,
    # metrics=[correctness, tone],
    metrics=[tone],
    async_config=AsyncConfig(max_concurrent=2),           # be gentle on Groq's rate limits
    display_config=DisplayConfig(print_results=True),
    error_config=ErrorConfig(ignore_errors=True, skip_on_missing_params=True),
)

✨ You're running DeepEval's latest Professional Tone [GEval] Metric! (using llama-3.3-70b-versatile (Local Model),
strict=False, async_mode=True)...

/Users/eshantdas/Desktop/SelfStudy/PersonalTest/KrishNaikUdemyLLMSecurity_gateways/.venv/lib/python3.13/site-packag
es/rich/live.py:260: UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_0 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  ❌ test_case_1                                                                                                 │
│  ├──   Input:              What's your refund policy?                                                           │
│  │     Actual Output:      No refunds. Read the policy page next time.                                          │
│  │     Expected Output:    Refunds are available within 30 days of purchase.                                    │
│  └── Metrics                                                                                                    │
│       Status ┃ Metric                    ┃ Score ┃ Threshold ┃ Reason                                           │
│      ━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  │
│        FAIL  │ Professional Tone [GEval] │ 0.20  │ 0.50      │ The response is brief but lacks politeness and   │
│              │                           │       │           │ professionalism, using dismissive language by    │
│              │                           │       │           │ saying 'next time', which is penalized           │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Aggregate Metrics                                                                                               │
│                                                                                                                 │
│  Metric                              ┃ Average Score       ┃ Pass Rate                               ┃ Total    │
│ ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━ │
│  Professional Tone [GEval]           │ 0.60                │ 50.00% | passed=1 | failed=1            │ 2        │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

⚠ WARNING: No hyperparameters logged.
» ]8;id=6487349;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 1.41s | token cost: None)
» Test Results (2 total tests):
   » Pass Rate: 50.0% | Passed: 1 | Failed: 1

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

In [9]:
results.test_results

[TestResult(name='test_case_1', success=False, metrics_data=[MetricData(name='Professional Tone [GEval]', threshold=0.5, success=False, score=0.2, reason="The response is brief but lacks politeness and professionalism, using dismissive language by saying 'next time', which is penalized", strict_mode=False, flaky=False, evaluation_model='llama-3.3-70b-versatile (Local Model)', error=None, evaluation_cost=0.0, input_tokens=0, output_tokens=0, verbose_logs='Criteria:\nNone \n \nEvaluation Steps:\n[\n    "Check whether the response is polite and professional.",\n    "Penalize sarcasm, rudeness, or dismissive language.",\n    "Reward clear, respectful phrasing even if brief."\n] \n \nRubric:\nNone \n \nScore: 0.2')], conversational=False, index=1, multimodal=False, input="What's your refund policy?", actual_output='No refunds. Read the policy page next time.', expected_output='Refunds are available within 30 days of purchase.', context=None, retrieval_context=None, turns=None, metadata=None

## CHecl excalDraw for Best Use Cases and Limitations

## DAG Custom Eval

In [10]:
import deepeval.metrics.base_metric as _base_metric
from deepeval.metrics.dag.schema import TaskNodeOutput
from pydantic import ValidationError


_original_resolve = _base_metric.resolve_template

def _patched_resolve(feature, class_name, method, **kwargs):
    if class_name == "BinaryJudgement" and method == "generate_non_binary_verdict":
        class_name = "NonBinaryJudgement"  # the real fix -- should never have said "Binary"
    return _original_resolve(feature, class_name, method, **kwargs)

_base_metric.resolve_template = _patched_resolve



## bug 2
_original_validate = TaskNodeOutput.model_validate.__func__

def _patched_validate(cls, obj, *args, **kwargs):
    try:
        return _original_validate(cls, obj, *args, **kwargs)
    except ValidationError:
        if isinstance(obj, dict) and isinstance(obj.get("output"), dict) and len(obj["output"]) == 1:
            unwrapped = {**obj, "output": next(iter(obj["output"].values()))}
            return _original_validate(cls, unwrapped, *args, **kwargs)
        raise

TaskNodeOutput.model_validate = classmethod(_patched_validate)


print("Patched.")

Patched.


In [11]:
from deepeval.metrics import DAGMetric
from deepeval.metrics.dag import (
    BinaryJudgementNode,
    DeepAcyclicGraph,
    NonBinaryJudgementNode,
    TaskNode,
    VerdictNode,
)
from deepeval.test_case import SingleTurnParams

In [12]:
def build_dag_metric():
    correct_order_node = NonBinaryJudgementNode(
        criteria="Are the summary headings in the correct order: 'intro' => 'body' => 'conclusion'?",
        children=[
            VerdictNode(verdict="Yes", score=10),
            VerdictNode(verdict="Two are out of order", score=4),
            VerdictNode(verdict="All out of order", score=2),
        ],
    )
    correct_headings_node = BinaryJudgementNode(
        criteria="Does the summary headings contain all three: 'intro', 'body', and 'conclusion'?",
        children=[
            VerdictNode(verdict=False, score=0),
            VerdictNode(verdict=True, child=correct_order_node),
        ],
    )
    extract_headings_node = TaskNode(
        instructions="Extract all headings in `actual_output`",
        evaluation_params=[SingleTurnParams.ACTUAL_OUTPUT],
        output_label="Summary headings",
        children=[correct_headings_node, correct_order_node],
    )

    dag = DeepAcyclicGraph(root_nodes=[extract_headings_node])
    return DAGMetric(name="Format Correctness", dag=dag, model=judge, async_mode=False)


In [13]:
meeting_transcript = (
    'Alice: "Today\'s agenda: product update, blockers, and marketing timeline. Bob, updates?"\n'
    'Bob: "Core features are done, but we\'re optimizing performance for large datasets. Fixes '
    'by Friday, testing next week."\n'
    'Alice: "Charlie, does this timeline work for marketing?"\n'
    'Charlie: "We need finalized messaging by Monday."\n'
    'Alice: "Bob, can we provide a stable version by then?"\n'
    'Bob: "Yes, we\'ll share an early build."\n'
    'Charlie: "Great, we\'ll start preparing assets."\n'
    'Alice: "Plan: fixes by Friday, marketing prep Monday, sync next Wednesday. Thanks, everyone!"'
)


In [14]:

dag_test_cases = [
    LLMTestCase(
        input=meeting_transcript,
        actual_output=(
            "Intro:\nAlice outlined the agenda: product updates, blockers, and marketing "
            "alignment.\n\nBody:\nBob reported performance issues being optimized, with fixes "
            "expected by Friday. Charlie requested finalized messaging by Monday for marketing "
            "preparation. Bob confirmed an early stable build would be ready.\n\nConclusion:\n"
            "The team aligned on next steps: engineering finalizing fixes, marketing preparing "
            "content, and a follow-up sync scheduled for Wednesday."
        ),
    ),
    LLMTestCase(
        input=meeting_transcript,
        actual_output=(
            "Intro:\nAlice outlined the agenda.\n\nBody:\nBob reported fixes by Friday and "
            "Charlie needs messaging by Monday."
        ),  # no Conclusion heading -- the gate should fail and skip the order check
    ),
]

## Run DAG and inspect the trace

In [15]:
for i, tc in enumerate(dag_test_cases, start=1):
    metric = build_dag_metric()
    metric.measure(tc)
    verdict = "PASS" if metric.is_successful() else "FAIL"
    raw_score = round(metric.score * 10)
    print(f"=== Test case {i}: [{verdict}] raw {raw_score}/10 -> normalized {metric.score:.2f} ===")
    print(metric.verbose_logs)
    print()

/Users/eshantdas/Desktop/SelfStudy/PersonalTest/KrishNaikUdemyLLMSecurity_gateways/.venv/lib/python3.13/site-packag
es/rich/live.py:260: UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

/var/folders/4d/gf6xkqsd5tx6m_zltry403dc0000gn/T/ipykernel_38488/4245401140.py:2: DeprecationWarning: Passing 'children' to 'NonBinaryJudgementNode' (the bottom-up way of building a DAG) is deprecated and will be removed in a future release. Construct 'NonBinaryJudgementNode' without 'children' and use 'add_verdict()' to build the DAG top-down instead.
  correct_order_node = NonBinaryJudgementNode(
/var/folders/4d/gf6xkqsd5tx6m_zltry403dc0000gn/T/ipykernel_38488/4245401140.py:10: DeprecationWarning: Passing 'children' to 'BinaryJudgementNode' (the bottom-up way of building a DAG) is deprecated and will be removed in a future release. Construct 'BinaryJudgementNode' without 'children' and use 'add_verdict()' to build the DAG top-down instead.
  correct_headings_node = BinaryJudgementNode(
/var/folders/4d/gf6xkqsd5tx6m_zltry403dc0000gn/T/ipykernel_38488/4245401140.py:17: DeprecationWarning: Passing 'children' to 'TaskNode' (the bottom-up way of building a DAG) is deprecated and will be r

=== Test case 1: [PASS] raw 10/10 -> normalized 1.00 ===
______________________
| TaskNode | Level == 0 |
*******************************
Label: None

Instructions:
Extract all headings in `actual_output`

Summary headings:
['Intro:', 'Body:', 'Conclusion:']
 
 
__________________________________
| BinaryJudgementNode | Level == 1 |
************************************************
Label: None

Criteria:
Does the summary headings contain all three: 'intro', 'body', and 'conclusion'?

Verdict: True
Reason: The summary headings contain all three required sections: 'Intro:', 'Body:', and 'Conclusion:', as specified.
 
 
_____________________________________
| NonBinaryJudgementNode | Level == 2 |
*****************************************************
Label: None

Criteria:
Are the summary headings in the correct order: 'intro' => 'body' => 'conclusion'?

Verdict: Yes
Reason: The summary headings are in the correct logical order, typically used in essay writing and other forms of writing: in

=== Test case 2: [FAIL] raw 0/10 -> normalized 0.00 ===
______________________
| TaskNode | Level == 0 |
*******************************
Label: None

Instructions:
Extract all headings in `actual_output`

Summary headings:
['Intro:', 'Body:']
 
 
__________________________________
| BinaryJudgementNode | Level == 1 |
************************************************
Label: None

Criteria:
Does the summary headings contain all three: 'intro', 'body', and 'conclusion'?

Verdict: False
Reason: The summary headings are missing 'conclusion'
 
 
________________________
| VerdictNode | Level == 2 |
**********************************
Verdict: False
Type: Deterministic



## RAG Evals using DeepEvals

In [16]:
CORPUS = [
    {"title": "Tokens", "content": (
        "Large language models split text into tokens, common character sequences roughly 4 "
        "characters or three-quarters of a word long. Both the prompt and the output are counted "
        "in tokens, and pricing and context limits are measured in tokens.")},
    {"title": "Embeddings", "content": (
        "An embedding is a fixed-length vector that represents the meaning of a piece of text. "
        "Texts with similar meaning have vectors that are close together, usually measured by "
        "cosine similarity, which is what lets a system do semantic search.")},
    {"title": "Retrieval-Augmented Generation", "content": (
        "RAG grounds an LLM's answer in external documents. At query time the system retrieves "
        "the most relevant chunks and passes them to the model as context, which reduces "
        "hallucination and lets you update knowledge without retraining.")},
    {"title": "Hallucination", "content": (
        "A hallucination is fluent, confident text that is factually wrong or unsupported by its "
        "sources. It happens because the model predicts likely text rather than looking facts up. "
        "Grounding answers in retrieved context is the main defense.")},
    {"title": "AI agents", "content": (
        "An AI agent is an LLM given a goal, tools, and a loop: plan, call a tool, observe the "
        "result, decide the next step, until the task is done.")},
]


In [17]:
def retrieve(query: str, k: int = 2) -> list[str]:
    """Naive retriever: rank docs by keyword overlap with the query. Good enough to teach evals."""
    q_words = set(query.lower().split())
    scored = []
    for doc in CORPUS:
        doc_words = set((doc["title"] + " " + doc["content"]).lower().split())
        overlap = len(q_words & doc_words)
        scored.append((overlap, doc))
    scored.sort(key=lambda x: x[0], reverse=True)
    return [doc["content"] for _, doc in scored[:k]]

In [18]:
### Simple RAG pipeline

In [19]:
from groq import Groq

groq_client = Groq()


RAG_SYSTEM = (
    "Answer ONLY using the provided context. If the context doesn't contain the answer, say you "
    "don't have that information. Be concise (1-2 sentences)."
)



def answer_with_groq(question: str, passages: list[str]) -> str:
    context = "\n\n".join(passages)
    prompt = f"Context:\n{context}\n\nQuestion: {question}\n\nAnswer:"
    resp = groq_client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[{"role": "system", "content": RAG_SYSTEM}, {"role": "user", "content": prompt}],
        temperature=0,
    )
    return resp.choices[0].message.content.strip()

In [20]:
QUESTIONS = [
    {"input": "What is a token in an LLM?",
     "expected_output": "A token is a common character sequence, roughly 4 characters, used to measure both prompt and output length."},
    {"input": "How does RAG reduce hallucination?",
     "expected_output": "RAG retrieves relevant chunks and passes them as context, grounding the answer in real documents instead of only the model's training."},
    {"input": "How do I containerize a model for deployment with Docker?"},  # not in our tiny KB
]


In [21]:

rag_rows = []
for q in QUESTIONS:
    passages = retrieve(q["input"])
    answer = answer_with_groq(q["input"], passages)
    rag_rows.append({**q, "actual_output": answer, "retrieval_context": passages})
    print("Q:", q["input"])
    print("A:", answer)
    print("-" * 80)

Q: What is a token in an LLM?
A: I don't have that information.
--------------------------------------------------------------------------------
Q: How does RAG reduce hallucination?
A: RAG reduces hallucination by grounding the model’s response in retrieved external document chunks, giving it factual context to base its answer on rather than generating unconstrained text.
--------------------------------------------------------------------------------
Q: How do I containerize a model for deployment with Docker?
A: I don't have that information.
--------------------------------------------------------------------------------


In [23]:
rag_rows

[{'input': 'What is a token in an LLM?',
  'expected_output': 'A token is a common character sequence, roughly 4 characters, used to measure both prompt and output length.',
  'actual_output': "I don't have that information.",
  'retrieval_context': ['An embedding is a fixed-length vector that represents the meaning of a piece of text. Texts with similar meaning have vectors that are close together, usually measured by cosine similarity, which is what lets a system do semantic search.',
   'A hallucination is fluent, confident text that is factually wrong or unsupported by its sources. It happens because the model predicts likely text rather than looking facts up. Grounding answers in retrieved context is the main defense.']},
 {'input': 'How does RAG reduce hallucination?',
  'expected_output': "RAG retrieves relevant chunks and passes them as context, grounding the answer in real documents instead of only the model's training.",
  'actual_output': 'RAG reduces hallucination by ground

### Creating LLMTestCases

In [22]:
from deepeval.test_case import LLMTestCase

test_cases = [
    LLMTestCase(
        input=r["input"],
        actual_output=r["actual_output"],
        expected_output=r.get("expected_output"),
        retrieval_context=r["retrieval_context"],
    )
    for r in rag_rows
]

### Performing Evals

In [24]:
from deepeval.models import LocalModel

judge = LocalModel(
    model="llama-3.1-8b-instant",
    api_key=os.environ["GROQ_API_KEY"],
    base_url="https://api.groq.com/openai/v1",
    temperature=0,
)
print("Judge ready:", judge.get_model_name())

Judge ready: llama-3.1-8b-instant (Local Model)


In [25]:
from deepeval.metrics import (
    AnswerRelevancyMetric,
    ContextualPrecisionMetric,
    ContextualRecallMetric,
    ContextualRelevancyMetric,
    FaithfulnessMetric,
)

metrics = [
    FaithfulnessMetric(model=judge),
    AnswerRelevancyMetric(model=judge),
    ContextualRelevancyMetric(model=judge),
    ContextualPrecisionMetric(model=judge),
    ContextualRecallMetric(model=judge),
]

In [26]:
from deepeval import evaluate
from deepeval.evaluate.configs import AsyncConfig, DisplayConfig, ErrorConfig

results = evaluate(
    test_cases=test_cases,
    metrics=metrics,
    async_config=AsyncConfig(max_concurrent=1, throttle_value=2.0),  # ~41 calls -- spread the burst
    display_config=DisplayConfig(print_results=True),
    error_config=ErrorConfig(ignore_errors=True, skip_on_missing_params=True),
)

✨ You're running DeepEval's latest Faithfulness Metric! (using llama-3.1-8b-instant (Local Model), strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Answer Relevancy Metric! (using llama-3.1-8b-instant (Local Model), 
strict=False, async_mode=True)...

✨ You're running DeepEval's latest Contextual Relevancy Metric! (using llama-3.1-8b-instant (Local Model), 
strict=False, async_mode=True)...

✨ You're running DeepEval's latest Contextual Precision Metric! (using llama-3.1-8b-instant (Local Model), 
strict=False, async_mode=True)...

✨ You're running DeepEval's latest Contextual Recall Metric! (using llama-3.1-8b-instant (Local Model), 
strict=False, async_mode=True)...

/Users/eshantdas/Desktop/SelfStudy/PersonalTest/KrishNaikUdemyLLMSecurity_gateways/.venv/lib/python3.13/site-packag
es/rich/live.py:260: UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  ❌ test_case_0                                                                                                 │
│  ├──   Input:              What is a token in an LLM?                                                           │
│  │     Actual Output:      I don't have that information.                                                       │
│  │     Expected Output:    A token is a common character sequence, roughly 4 characters, used to measure        │
│  │                         both prompt and output length.                                                       │
│  └── Metrics                                                                                                    │
│       Status ┃ Metric               ┃ Score ┃ Threshold ┃ Reason                                                │
│      ━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  │
│       ERROR  │ Faithfulness         │ N/A   │ 0.50      │ RetryError[<Future at 0x11894f850 state=finished      │
│              │                      │       │           │ raised RateLimitError>]                               │
│        FAIL  │ Answer Relevancy     │ 0.00  │ 0.50      │ The score is 0.00 because the actual output           │
│              │                      │       │           │ provided a generic response that does not address     │
│              │                      │       │           │ the question about tokens in an LLM, making it        │
│              │                      │       │           │ completely irrelevant to the input.                   │
│        PASS  │ Contextual Relevancy │ 1.00  │ 0.50      │ The score is 1.00 because the retrieval context...    │
│        FAIL  │ Contextual Precision │ 0.00  │ 0.50      │ The score is 0.00 because the first two nodes in      │
│              │                      │       │           │ the retrieval contexts are irrelevant nodes, ranked   │
│              │                      │       │           │ higher than the relevant nodes, with reason 'This     │
│              │                      │       │           │ context does not mention tokens, so it's not          │
│              │                      │       │           │ relevant to the question.' for both nodes.            │
│        FAIL  │ Contextual Recall    │ 0.25  │ 0.50      │ The score is 0.25 because the expected output has     │
│              │                      │       │           │ some relevant information about measuring             │
│              │                      │       │           │ similarity (sentence 1), but the rest of the          │
│              │                      │       │           │ sentences (sentences 2-3) do not match any text in    │
│              │                      │       │           │ the node(s) in retrieval context, specifically node   │
│              │                      │       │           │ 1, which mentions cosine similarity, but does not     │
│              │                      │       │           │ provide enough context to support the entire          │
│              │                      │       │           │ expected output.                                      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────

⚠ WARNING: No hyperparameters logged.
» ]8;id=6487351;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 12.43s | token cost: None)
» Test Results (3 total tests):
   » Pass Rate: 0.0% | Passed: 0 | Failed: 3

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

In [28]:
results.test_results

[TestResult(name='test_case_0', success=False, metrics_data=[MetricData(name='Faithfulness', threshold=0.5, success=False, score=None, reason=None, strict_mode=False, flaky=False, evaluation_model='llama-3.1-8b-instant (Local Model)', error='RetryError[<Future at 0x11894f850 state=finished raised RateLimitError>]', evaluation_cost=0.0, input_tokens=0, output_tokens=0, verbose_logs=None), MetricData(name='Answer Relevancy', threshold=0.5, success=False, score=0.0, reason='The score is 0.00 because the actual output provided a generic response that does not address the question about tokens in an LLM, making it completely irrelevant to the input.', strict_mode=False, flaky=False, evaluation_model='llama-3.1-8b-instant (Local Model)', error=None, evaluation_cost=0.0, input_tokens=0, output_tokens=0, verbose_logs='Statements:\n[\n    "I don\'t have that information."\n] \n \nVerdicts:\n[\n    {\n        "verdict": "no",\n        "reason": "The statement is a generic response and does not p